In [1]:
import pandas as pd

books = pd.read_csv("../data/books_cleaned.csv")

In [3]:
books["categories"].value_counts().reset_index()

,categories,count
0,Exhibitions,10
1,Literature,8
2,"Fiction, romance, general",8
3,English literature,7
4,American literature,7
...,...,...
4719,"Fiction, family life;Brothers, fiction;India, ...",1
4720,Taiwan aborigines;Ethnobotany;Ethnozoology;Mus...,1
4721,Fiction;Aircraft accidents;Large type books;Fi...,1
4722,Actors;Cats;Fiction;Metamorphosis;Large type b...,1


In [4]:
# get rid of categories with less than 50 books
books["categories"].value_counts().reset_index().query("count > 50")

,categories,count


In [7]:
import re

# OL puts many subjects into one ';'-joined string so scan for keywords. 
# "nonfiction" contains "fiction", so nonfiction tested before fiction.

# fuzzier genre hints consulted when no explicit fiction/nonfiction label is present.
FICTION_HINTS = (
    "fantasy", "romance", "thriller", "mystery", "horror", "science fiction",
    "short stories", "fairy tale", "graphic novel", "comic", "detective",
    "adventure", "poetry", "drama", "novel",
)
NONFICTION_HINTS = (
    "biography", "autobiography", "history", "philosophy", "religion", "self-help",
    "cooking", "travel", "business", "psychology", "memoir", "true crime", "essays",
    "science", "reference", "health",
)

def simplify_categories(cats):
    if not isinstance(cats, str):
        return None
    c = cats.lower()
    juvenile = ("juvenile" in c) or ("children" in c)

    if re.search(r"non-?fiction", c):              # 1: explicit labels
        kind = "Nonfiction"
    elif re.search(r"\bfiction\b", c):
        kind = "Fiction"
    elif any(h in c for h in FICTION_HINTS):       # 2: genre hints
        kind = "Fiction"
    elif any(h in c for h in NONFICTION_HINTS):
        kind = "Nonfiction"
    else:
        return None                                

    return f"Children's {kind}" if juvenile else kind

books["simple_categories"] = books["categories"].apply(simplify_categories)
books["simple_categories"].value_counts(dropna=False)

simple_categories
Nonfiction               1785
NaN                      1658
Fiction                  1084
Children's Fiction        406
Children's Nonfiction      67
Name: count, dtype: int64

In [8]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9780349701462,Vanishing Half,Brit Bennett,African American;Twins;fiction;race;identity;c...,"Brit Bennett’s chart topping novel, The Vanish...",https://covers.openlibrary.org/b/id/10680969-L...,2020,NaN,352.0,Vanishing Half: A Novel,9780349701462 Brit Bennett’s chart topping nov...,Fiction
1,9780062853509,Ghost Radio,Leopoldo Gout,Fiction;Radio broadcasters in fiction;Radio ta...,Ghost Radio is a terrifying novel about a ghos...,NaN,2018,NaN,NaN,Ghost Radio: A Novel,9780062853509 Ghost Radio is a terrifying nove...,Fiction
2,9780567688682,Exodus 1-18,Graham I. Davies;Graham I. Davies;Christopher ...,"Bible, commentaries, o. t. pentateuch;Bible;Co...","""Graham I. Davies provides his long-awaited co...",NaN,2020,NaN,816.0,Exodus 1-18: A Critical and Exegetical Commentary,"9780567688682 ""Graham I. Davies provides his l...",NaN
3,9780807507896,Skeleton Key Mystery,Gertrude Chandler Warner;Anthony VanArsdale,"Adventure and adventurers, fiction;Brothers an...",The Aldens are visiting a small town known for...,https://covers.openlibrary.org/b/id/13770184-L...,2020,NaN,128.0,Skeleton Key Mystery,9780807507896 The Aldens are visiting a small ...,Children's Fiction
4,9781944316174,Matthew Wong,Matthew Wong;Cheim & Read,Exhibitions;Art,"Over the course of his brief career, Matthew W...",NaN,2021,NaN,NaN,Matthew Wong: footprints in the wind : ink dra...,9781944316174 Over the course of his brief car...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9784906962860,Environmental teachings for the anthropocene,Atsushi Nobayashi;Scott Simon,Taiwan aborigines;Ethnobotany;Ethnozoology;Mus...,"""The essays in this volume are organised into ...",NaN,2020,NaN,230.0,Environmental teachings for the anthropocene: ...,"9784906962860 ""The essays in this volume are o...",NaN
4996,9781420128840,"Night, Sea and Stars",Heather Graham,Fiction;Aircraft accidents;Large type books;Fi...,Crash-landing on a remote South Pacific island...,NaN,2020,NaN,352.0,"Night, Sea and Stars",9781420128840 Crash-landing on a remote South ...,Fiction
4997,9781504058582,Nine Lives to Murder,Marian Babson,Actors;Cats;Fiction;Metamorphosis;Large type b...,"Marion Babson always seems to write funny, lig...",NaN,2019,NaN,NaN,Nine Lives to Murder,9781504058582 Marion Babson always seems to wr...,Fiction
4998,9789492051448,Zilverbeek,Lucas Leffler,Artistic Photography;Pictorial works,Since the 1920s the Belgian factory Gevaert ac...,NaN,2019,NaN,NaN,Zilverbeek: Silver Creek,9789492051448 Since the 1920s the Belgian fact...,NaN


In [9]:
books[~(books["simple_categories"].isna())]

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9780349701462,Vanishing Half,Brit Bennett,African American;Twins;fiction;race;identity;c...,"Brit Bennett’s chart topping novel, The Vanish...",https://covers.openlibrary.org/b/id/10680969-L...,2020,NaN,352.0,Vanishing Half: A Novel,9780349701462 Brit Bennett’s chart topping nov...,Fiction
1,9780062853509,Ghost Radio,Leopoldo Gout,Fiction;Radio broadcasters in fiction;Radio ta...,Ghost Radio is a terrifying novel about a ghos...,NaN,2018,NaN,NaN,Ghost Radio: A Novel,9780062853509 Ghost Radio is a terrifying nove...,Fiction
3,9780807507896,Skeleton Key Mystery,Gertrude Chandler Warner;Anthony VanArsdale,"Adventure and adventurers, fiction;Brothers an...",The Aldens are visiting a small town known for...,https://covers.openlibrary.org/b/id/13770184-L...,2020,NaN,128.0,Skeleton Key Mystery,9780807507896 The Aldens are visiting a small ...,Children's Fiction
6,9781032238876,Double Trouble,Eran Dorfman,Doubles in literature;Difference (philosophy);...,"""The double, doppelga nger, is mostly understo...",NaN,2021,NaN,NaN,Double Trouble,"9781032238876 ""The double, doppelga nger, is m...",Fiction
7,9780062850218,Galaxy girls,Libby Jackson,Exploration;Women in computer science;Women in...,Filled with beautiful full-color illustrations...,https://covers.openlibrary.org/b/id/12030171-L...,2018,NaN,143.0,Galaxy girls: 50 amazing stories of women in s...,9780062850218 Filled with beautiful full-color...,Children's Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...
4991,9780755602735,Myth Making in the Soviet Union and Modern Russia,Vicky Davis,"Brezhnev, leonid i., 1906-1982;World war, 1939...",The 1943 battle to free the Soviet Black Sea p...,NaN,2020,NaN,NaN,Myth Making in the Soviet Union and Modern Rus...,9780755602735 The 1943 battle to free the Sovi...,Nonfiction
4992,9780735229266,The Mad Wolf's daughter,Diane Magras,Adventure and adventurers;Family life;Fiction;...,"""In 1210 Scotland, when invading knights captu...",https://covers.openlibrary.org/b/id/12554322-L...,2018,NaN,280.0,The Mad Wolf's daughter,"9780735229266 ""In 1210 Scotland, when invading...",Children's Fiction
4994,9781509886548,Selection Day,Aravind Adiga,"Fiction, family life;Brothers, fiction;India, ...","From Aravind Adiga, the bestselling, Booker Pr...",NaN,2018,NaN,352.0,Selection Day: TV Tie-In,"9781509886548 From Aravind Adiga, the bestsell...",Fiction
4996,9781420128840,"Night, Sea and Stars",Heather Graham,Fiction;Aircraft accidents;Large type books;Fi...,Crash-landing on a remote South Pacific island...,NaN,2020,NaN,352.0,"Night, Sea and Stars",9781420128840 Crash-landing on a remote South ...,Fiction


In [10]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="mps")
# mps is the Apple Mac specific GPU

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [11]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]
# resetting the index ensure indexes correlate to the classified books not the original database

In [12]:
pipe(sequence, fiction_categories)
# returns probability that the book is within each category

{'sequence': 'Brit Bennett’s chart topping novel, The Vanishing Half, is a story that tracks the lives of twin African American twin sisters who, after witnessing the murder of their father, run away at age 16. One sister begins passing as white and the other sister remains true to her identity. The Vanishing Half explores the intricacies of identity, family, and race in a provocative, but compassionate way.',
 'labels': ['Fiction', 'Nonfiction'],
 'scores': [0.9505485892295837, 0.04945140704512596]}

In [13]:
import numpy as np

max_index = np.argmax(pipe(sequence, fiction_categories)["scores"])
max_label = pipe(sequence, fiction_categories)["labels"][max_index]
max_label

'Fiction'

In [14]:
def generate_predictions(sequence, categories):
    predictions = pipe(sequence, categories)
    max_index = np.argmax(predictions["scores"]) # yields the index of the highest probability
    max_label = predictions["labels"][max_index]
    return max_label

In [15]:
# Testing how good the model is using a sizable sample
from tqdm import tqdm
# tqdm is a library used to add progress bars to loops

actual_cats = []
predicted_cats = []

for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]

100%|██████████| 100/100 [00:17<00:00,  5.61it/s]


In [16]:
for i in tqdm(range(0, 100)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]

100%|██████████| 100/100 [00:14<00:00,  6.75it/s]


In [17]:
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predicted_categories": predicted_cats})
predictions_df

,actual_categories,predicted_categories
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Nonfiction
3,Fiction,Fiction
4,Fiction,Nonfiction
...,...,...
195,Nonfiction,Nonfiction
196,Nonfiction,Nonfiction
197,Nonfiction,Nonfiction
198,Nonfiction,Nonfiction


In [18]:
predictions_df["correct_prediction"] = (
    np.where(predictions_df["actual_categories"] == predictions_df["predicted_categories"], 1, 0)
)

In [ ]:
# Percentage accuracy of the BART zero-shot Fiction/Nonfiction backfill on the 200-book labelled sample.
# TODO: improve in future
predictions_df["correct_prediction"].sum() / len(predictions_df)

np.float64(0.7)

In [20]:
# make a subset of the database with the simple category missing
isbns = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)

In [21]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbns += [missing_cats["isbn13"][i]]

100%|██████████| 1658/1658 [03:33<00:00,  7.76it/s]


In [22]:
missing_predicted_df = pd.DataFrame({"isbn13": isbns, "predicted_categories": predicted_cats})

In [23]:
missing_predicted_df

,isbn13,predicted_categories
0,9780567688682,Nonfiction
1,9781944316174,Nonfiction
2,9788193986608,Nonfiction
3,9781108735872,Nonfiction
4,9781350005501,Nonfiction
...,...,...
1653,9781946433602,Fiction
1654,9781683505679,Nonfiction
1655,9784906962860,Nonfiction
1656,9789492051448,Nonfiction


In [24]:
books = pd.merge(books, missing_predicted_df, on="isbn13", how="left") 
books["simple_categories"] = np.where(books["simple_categories"].isna(), books["predicted_categories"], books["simple_categories"])
# merge into books - when category is missing use predicted, if simple category there then use that
books = books.drop(columns = ["predicted_categories"])

In [25]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories
0,9780349701462,Vanishing Half,Brit Bennett,African American;Twins;fiction;race;identity;c...,"Brit Bennett’s chart topping novel, The Vanish...",https://covers.openlibrary.org/b/id/10680969-L...,2020,NaN,352.0,Vanishing Half: A Novel,9780349701462 Brit Bennett’s chart topping nov...,Fiction
1,9780062853509,Ghost Radio,Leopoldo Gout,Fiction;Radio broadcasters in fiction;Radio ta...,Ghost Radio is a terrifying novel about a ghos...,NaN,2018,NaN,NaN,Ghost Radio: A Novel,9780062853509 Ghost Radio is a terrifying nove...,Fiction
2,9780567688682,Exodus 1-18,Graham I. Davies;Graham I. Davies;Christopher ...,"Bible, commentaries, o. t. pentateuch;Bible;Co...","""Graham I. Davies provides his long-awaited co...",NaN,2020,NaN,816.0,Exodus 1-18: A Critical and Exegetical Commentary,"9780567688682 ""Graham I. Davies provides his l...",Nonfiction
3,9780807507896,Skeleton Key Mystery,Gertrude Chandler Warner;Anthony VanArsdale,"Adventure and adventurers, fiction;Brothers an...",The Aldens are visiting a small town known for...,https://covers.openlibrary.org/b/id/13770184-L...,2020,NaN,128.0,Skeleton Key Mystery,9780807507896 The Aldens are visiting a small ...,Children's Fiction
4,9781944316174,Matthew Wong,Matthew Wong;Cheim & Read,Exhibitions;Art,"Over the course of his brief career, Matthew W...",NaN,2021,NaN,NaN,Matthew Wong: footprints in the wind : ink dra...,9781944316174 Over the course of his brief car...,Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9784906962860,Environmental teachings for the anthropocene,Atsushi Nobayashi;Scott Simon,Taiwan aborigines;Ethnobotany;Ethnozoology;Mus...,"""The essays in this volume are organised into ...",NaN,2020,NaN,230.0,Environmental teachings for the anthropocene: ...,"9784906962860 ""The essays in this volume are o...",Nonfiction
4996,9781420128840,"Night, Sea and Stars",Heather Graham,Fiction;Aircraft accidents;Large type books;Fi...,Crash-landing on a remote South Pacific island...,NaN,2020,NaN,352.0,"Night, Sea and Stars",9781420128840 Crash-landing on a remote South ...,Fiction
4997,9781504058582,Nine Lives to Murder,Marian Babson,Actors;Cats;Fiction;Metamorphosis;Large type b...,"Marion Babson always seems to write funny, lig...",NaN,2019,NaN,NaN,Nine Lives to Murder,9781504058582 Marion Babson always seems to wr...,Fiction
4998,9789492051448,Zilverbeek,Lucas Leffler,Artistic Photography;Pictorial works,Since the 1920s the Belgian factory Gevaert ac...,NaN,2019,NaN,NaN,Zilverbeek: Silver Creek,9789492051448 Since the 1920s the Belgian fact...,Nonfiction


In [27]:
books.to_csv("../data/books_with_categories.csv", index=False)